In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

from agx_core.models.ra.encoder import Encoder
from agx_core.models.ra.decoder import Decoder
from agx_torch.models.ra.model import ReversedAutoencoder

In [ ]:
enc = Encoder()

img_size = 192
res = img_size // 2 ** len(enc.filters)

img_shape = (img_size, img_size, 1)
cond_shape = (res, res, 1)

dec = Decoder(target_shape=img_shape)
ra = ReversedAutoencoder(enc, dec, scale=0.05)

In [ ]:
import torch
import numpy as np
import albumentations as A

from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset, DataLoader


class UnlabeledImageDataset(Dataset):
    def __init__(self, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        # Get list of all image file names in the folder
        self.image_files = list(root_dir.glob("*.jpg"))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(img_name).convert("L")

        if self.transform:
            image = self.transform(image=np.array(image))
            image = image["image"][..., np.newaxis]

        condition = np.ones(cond_shape, dtype=np.int64)
        return (image, condition), image.copy()


train_path = Path("../data/products/TunaZenithal/train/")
valid_path = Path("../data/products/TunaZenithal/val/")

tx_train = A.Compose(
    [
        A.InvertImg(1),
        A.Resize(img_size, img_size),
        A.HorizontalFlip(0.5),
        # A.Affine(scale=(0.9, 0.95), rotate=(15, 90), shear=(-2, 2), p=0.5),
        A.RandomRotate90(0.5),
        # A.RandomBrightnessContrast(
        #     brightness_limit=0.2, contrast_limit=(-0.5, 0.5), p=0.5
        # ),
        A.GaussianBlur(blur_limit=(1, 3), p=0.3),
        A.ToFloat(255),
    ]
)

tx_valid = A.Compose(
    [
        A.InvertImg(1),
        A.Resize(img_size, img_size),
        A.ToFloat(255),
    ]
)

ds_train = UnlabeledImageDataset(train_path, transform=tx_train)
ds_valid = UnlabeledImageDataset(valid_path, transform=tx_valid)

loader_train = DataLoader(ds_train, batch_size=8, shuffle=True)
loader_valid = DataLoader(ds_valid, batch_size=8)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for idx, ax in enumerate(axes.flat):
    X, y = ds_train[idx]
    ax.imshow(X[0], cmap="gray")
    ax.set_title(f"Image {idx + 1}")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from agx_core.models.ra.optimizer import RAOptimizer

from keras.optimizers import Adam
from keras.losses import  MeanSquaredError

optimizer = RAOptimizer(Adam(learning_rate=1e-9), Adam(learning_rate=6e-6))
loss = MeanSquaredError(reduction=None)

ra.build([img_shape, cond_shape])
ra.compile(optimizer, loss)

In [ ]:
# history = ra.fit(loader_train, validation_data=loader_valid, epochs=20)

# fig, axes = plt.subplots(3, 3, figsize=(12, 12))

# for idx, ax in enumerate(axes.flat):
#     (I, C), y = ds_valid[idx]
#     rec = ra([I[np.newaxis, ...], C[np.newaxis, ...]])
#     ax.imshow(rec.cpu()[0], cmap="gray")
#     ax.set_title(f"Image {idx + 1}")
#     ax.axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
device = torch.device("cpu")
ra.eval()
ra = ra.to(device)
I, C = ds_valid[0][0]
I, C = torch.tensor(I[np.newaxis, ...], device=device), torch.tensor(
    C[np.newaxis, ...], device=device
)

torch.onnx.export(
    ra,
    ([I, C],),
    "model.onnx",
    input_names=["image", "condition"],
    output_names=["reconstruction"],
)